In [ ]:
# ============================
# CT Lung Cancer (Binary) — FULL IN ONE CELL
# Normal(0) vs Lung Cancer(1)
# - EfficientNetB0 baseline
# - No class_weight (avoids collapse on small datasets)
# - Threshold selected on VAL for TARGET_RECALL (medical screening)
# - Full eval: ROC-AUC, PR-AUC, report, Confusion Matrix (Blues) + ROC/PR + hist
# ============================

import os, random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, accuracy_score
)

# ---------- Config ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_ROOT = r"/path/to/local/resource CANCER"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

CLASS_NAMES = ["Normal", "Lung Cancer"]  # IMPORTANT: 0=Normal, 1=Cancer

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
EPOCHS = 15

# Medical operating point (screening): choose threshold on VAL to hit recall target
TARGET_RECALL = 0.90

# ---------- Sanity checks ----------
assert os.path.exists(TRAIN_DIR), f"Train dir not found: {TRAIN_DIR}"
assert os.path.exists(TEST_DIR),  f"Test dir not found:  {TEST_DIR}"
for c in CLASS_NAMES:
    assert os.path.exists(os.path.join(TRAIN_DIR, c)), f"Missing train class folder: {c}"
    assert os.path.exists(os.path.join(TEST_DIR, c)),  f"Missing test class folder: {c}"
print("✅ Folders OK")

# ---------- Build datasets ----------
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.15,
    subset="training",
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.15,
    subset="validation",
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("✅ class_names (raw):", train_ds_raw.class_names)

# ---------- Preprocess ----------
def norm_gray3(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    # CT slices are often grayscale stored as RGB; enforce stable grayscale->RGB
    x = tf.image.rgb_to_grayscale(x)
    x = tf.image.grayscale_to_rgb(x)
    return x, y

# (Optional) light augmentation; you can disable for CT by commenting apply_aug()
aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.02),
    tf.keras.layers.RandomZoom(0.05),
], name="aug")

def apply_aug(x, y):
    return aug(x, training=True), y

train_ds = train_ds_raw.map(norm_gray3, num_parallel_calls=AUTOTUNE).map(apply_aug, num_parallel_calls=AUTOTUNE)
val_ds   = val_ds_raw.map(norm_gray3, num_parallel_calls=AUTOTUNE)
test_ds  = test_ds_raw.map(norm_gray3, num_parallel_calls=AUTOTUNE)

# Prefetch + ignore bad files
train_ds = train_ds.prefetch(AUTOTUNE).apply(tf.data.experimental.ignore_errors())
val_ds   = val_ds.prefetch(AUTOTUNE).apply(tf.data.experimental.ignore_errors())
test_ds  = test_ds.prefetch(AUTOTUNE).apply(tf.data.experimental.ignore_errors())

# ---------- Quick class counts (train) ----------
y_train = []
for _, y in train_ds_raw.unbatch():
    y_train.append(int(y.numpy()))
y_train = np.array(y_train)
print("Train counts => Normal(0):", int((y_train==0).sum()), "Cancer(1):", int((y_train==1).sum()))

# ---------- Model ----------
base = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    pooling="avg",
)
base.trainable = False

inp = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = base(inp, training=False)
x = tf.keras.layers.Dropout(0.3)(x)

# Bias init helps calibration (prevents weird constant outputs)
pos_rate = float((y_train==1).mean())
bias = np.log(pos_rate / (1.0 - pos_rate + 1e-9))
out = tf.keras.layers.Dense(1, activation="sigmoid",
                            bias_initializer=tf.keras.initializers.Constant(bias))(x)

model = tf.keras.Model(inp, out)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)

model.summary()

# ---------- Train ----------
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=2, min_lr=1e-6),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

# ---------- Collect probs ----------
def collect_probs(ds):
    y_true, y_prob = [], []
    for x, y in ds:
        p = model.predict(x, verbose=0).reshape(-1)
        y_prob.append(p)
        y_true.append(y.numpy().reshape(-1))
    y_true = np.concatenate(y_true).astype(int)
    y_prob = np.concatenate(y_prob).astype(float)
    return y_true, y_prob

y_val, p_val   = collect_probs(val_ds)
y_test, p_test = collect_probs(test_ds)

print("p_test min/mean/max:", float(p_test.min()), float(p_test.mean()), float(p_test.max()))
print("Normal mean:", float(p_test[y_test==0].mean()), "Cancer mean:", float(p_test[y_test==1].mean()))

# ---------- Plot prob distribution ----------
plt.figure(figsize=(7,5))
plt.hist(p_test[y_test==0], bins=30, alpha=0.7, label="Normal")
plt.hist(p_test[y_test==1], bins=30, alpha=0.7, label="Cancer")
plt.axvline(0.5, linestyle="--")
plt.legend()
plt.xlabel("Predicted probability (Cancer)")
plt.ylabel("Count")
plt.title("Test probability distribution")
plt.tight_layout()
plt.show()

# ---------- Pick threshold for TARGET_RECALL on VAL ----------
def pick_thr_recall(y, p, target=0.90):
    # Search a sensible range for small probabilities too
    for t in np.linspace(0.99, 0.001, 2000):
        yhat = (p >= t).astype(int)  # 1=Cancer
        tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
        rec = tp/(tp+fn+1e-9)
        spec = tn/(tn+fp+1e-9)
        if rec >= target:
            return float(t), float(rec), float(spec), (tn, fp, fn, tp)
    return 0.5, None, None, None

thr, val_rec, val_spec, val_stats = pick_thr_recall(y_val, p_val, TARGET_RECALL)
print("Chosen threshold:", thr, "| VAL recall:", val_rec, "| VAL spec:", val_spec, "| VAL stats (tn,fp,fn,tp):", val_stats)

# ---------- Evaluate on TEST ----------
auc = roc_auc_score(y_test, p_test)
ap  = average_precision_score(y_test, p_test)

y_pred = (p_test >= thr).astype(int)  # 1=Cancer
pred_rate = float(y_pred.mean())
cm = confusion_matrix(y_test, y_pred, labels=[0,1])
tn, fp, fn, tp = cm.ravel()

test_acc = accuracy_score(y_test, y_pred)
test_rec = tp/(tp+fn+1e-9)
test_spec = tn/(tn+fp+1e-9)
test_prec = tp/(tp+fp+1e-9)

print("\n==== TEST METRICS ====")
print("ROC-AUC:", float(auc))
print("PR-AUC :", float(ap))
print("Accuracy:", float(test_acc))
print("Precision:", float(test_prec))
print("Recall (Sensitivity):", float(test_rec))
print("Specificity:", float(test_spec))
print("Pred cancer rate:", pred_rate)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Normal","Cancer"], digits=4))

# ---------- Confusion Matrix (Blues) ----------
plt.figure(figsize=(6,5))
plt.imshow(cm, cmap="Blues")
plt.title(f"CT Lung Cancer — Confusion Matrix (thr={thr:.3f})")
plt.xticks([0,1], ["Normal(0)","Cancer(1)"])
plt.yticks([0,1], ["Normal(0)","Cancer(1)"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i,j]),
                 ha="center", va="center",
                 color="white" if cm[i,j] > cm.max()/2 else "black")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

# ---------- ROC & PR curves ----------
fpr, tpr, _ = roc_curve(y_test, p_test)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("CT Lung Cancer — ROC")
plt.legend()
plt.tight_layout()
plt.show()

prec_curve, rec_curve, _ = precision_recall_curve(y_test, p_test)
plt.figure(figsize=(6,5))
plt.plot(rec_curve, prec_curve, label=f"AP={ap:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("CT Lung Cancer — Precision-Recall")
plt.legend()
plt.tight_layout()
plt.show()
